In [1]:
import pandas as pd
import json
from pathlib import Path


In [2]:
DATA_DIR = Path("D:/AI_BRCA/DATOS_CANCER")
METADATA_FILE = DATA_DIR / "metadata.cancer.json"

In [3]:
# Leemos el JSON para asociar el nombre del archivo TSV con el 'case submitter id' (ej. TCGA-A1-A0SB-01A)
with open(METADATA_FILE, 'r') as f:
    metadata = json.load(f)

file_to_sample = {}
for item in metadata:
    filename = item.get("file_name")
    # Extraemos el ID del submitter. Tomamos los primeros 15 caracteres para identificar el Sample (ej. TCGA-XX-XXXX-01A)
    # Si quieres el paciente a nivel global sin importar el tipo de muestra, usa [:12] (ej. TCGA-XX-XXXX)
    sample_id = item["associated_entities"][0]["entity_submitter_id"][:15]
    file_to_sample[filename] = sample_id

In [4]:
counts_data = {}
tpm_data = {}

# Buscamos de forma recursiva todos los archivos TSV en la carpeta y subcarpetas
tsv_files = list(DATA_DIR.rglob("*.tsv"))
print(f"Encontrados {len(tsv_files)} archivos TSV. Procesando...")

for tsv_path in tsv_files:
    filename = tsv_path.name
    
    # Ignoramos archivos TSV que no estén en nuestro metadata
    if filename not in file_to_sample:
        continue
        
    sample_id = file_to_sample[filename]
    
    # Leer el TSV. Los archivos GDC suelen saltarse los comentarios si existen con comment='#'
    # Las columnas estándar son: gene_id, gene_name, gene_type, unstranded, stranded_first, stranded_second, tpm_unstranded, etc.
    df = pd.read_csv(tsv_path, sep='\t', comment='#')
    
    # Filtrar metadatos del secuenciador que GDC pone al inicio (N_unmapped, N_multimapping, etc.)
    # Nos quedamos solo con las filas donde el gene_id empiece con 'ENSG'
    df = df[df['gene_id'].str.startswith('ENSG', na=False)]
    
    # Algunos 'gene_name' pueden estar duplicados debido a múltiples Ensembl IDs apuntando al mismo símbolo HGNC.
    # Para Machine Learning / DEA, lo estándar es sumar estos conteos.
    df_grouped = df.groupby('gene_name')[['unstranded', 'tpm_unstranded']].sum()
    
    # Guardamos las Series en nuestros diccionarios usando el sample_id como llave
    counts_data[sample_id] = df_grouped['unstranded']
    tpm_data[sample_id] = df_grouped['tpm_unstranded']

Encontrados 120 archivos TSV. Procesando...


In [5]:
# pd.DataFrame(dict) pone las llaves (samples) como columnas y los índices (genes) como filas.
# Usamos .T (Transpuesta) porque en ML y PyDESeq2 se estila: Filas = Samples, Columnas = Genes/Features.
df_counts = pd.DataFrame(counts_data).T
df_tpm = pd.DataFrame(tpm_data).T

# Rellenamos posibles NaNs con 0 en caso de que algún gen falte en alguna muestra
df_counts = df_counts.fillna(0).astype(int)
df_tpm = df_tpm.fillna(0.0)

In [6]:
df_counts.to_csv("../data/processed/tcga_brca_raw_counts_CANCER.csv")
df_tpm.to_csv("../data/processed/tcga_brca_tpm_CANCER.csv")

In [7]:
vis = pd.read_csv("../data/processed/tcga_brca_raw_counts_CANCER.csv", index_col=0)
vis.head()


,5S_rRNA,5_8S_rRNA,7SK,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,...,ZYG11A,ZYG11AP1,ZYG11B,ZYX,ZYXP1,ZZEF1,ZZZ3,hsa-mir-1253,hsa-mir-423,snoZ196
TCGA-E2-A14U-01,5,0,0,19,192,1,51098,77,39,2,...,501,0,2410,16753,0,5633,3718,0,0,7
TCGA-E9-A1R3-01,3,0,0,17,81,2,58166,153,34,0,...,136,0,3835,9952,0,6443,3645,0,0,0
TCGA-B6-A408-01,1,0,0,30,190,1,22420,104,17,0,...,535,0,2014,9096,0,3174,2346,0,0,2
TCGA-AO-A0J8-01,0,0,0,13,180,0,28321,52,42,1,...,694,0,3970,9587,0,4599,5900,0,0,3
TCGA-BH-A1F0-01,3,0,0,13,127,6,105405,151,52,1,...,748,0,7817,15154,0,6904,5944,0,0,3


In [8]:
vis1 = pd.read_csv("../data/processed/tcga_brca_tpm_CANCER.csv", index_col=0)
vis1.head()

,5S_rRNA,5_8S_rRNA,7SK,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,...,ZYG11A,ZYG11AP1,ZYG11B,ZYX,ZYXP1,ZZEF1,ZZZ3,hsa-mir-1253,hsa-mir-423,snoZ196
TCGA-E2-A14U-01,2.7413,0.0,0.0,0.2823,3.3809,0.0062,480.5141,1.5518,0.3238,0.2629,...,6.3413,0.0,17.4489,304.6669,0.0,21.0646,24.5224,0.0,0.0,4.6729
TCGA-E9-A1R3-01,1.3523,0.0,0.0,0.2031,1.1471,0.0100,439.9034,2.4799,0.2270,0.0000,...,1.3844,0.0,22.3307,145.5556,0.0,19.3770,19.3347,0.0,0.0,0.0000
TCGA-B6-A408-01,0.4303,0.0,0.0,0.3841,2.8833,0.0053,181.6941,1.8063,0.1216,0.0000,...,5.8357,0.0,12.5665,142.5562,0.0,10.2287,13.3347,0.0,0.0,1.1506
TCGA-AO-A0J8-01,0.0000,0.0,0.0,0.1535,2.5191,0.0000,211.6677,0.8329,0.2771,0.1045,...,6.9814,0.0,22.8447,138.5668,0.0,13.6685,30.9278,0.0,0.0,1.5917
TCGA-BH-A1F0-01,0.9421,0.0,0.0,0.1082,1.2530,0.0208,555.3693,1.7051,0.2419,0.0736,...,5.3047,0.0,31.7109,154.4111,0.0,14.4654,21.9660,0.0,0.0,1.1221
